In [2]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as PILImage
from IPython.display import Image

import sys
PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.data_formatter as data_formatter
import multiomic_transformer.utils.experiment_handler as experiment_handler

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

GROUND_TRUTH_DIR = PROJECT_DIR / "data" / "ground_truth_files"
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA")

# Testing how the data prep and caching interacts with the experiment_loader class

## Load a Saved TrainingDataFormatter Object

We can load a TDF object from the disk using the `load_tdf()` function. This recreates the TDF object from a cached `settings.json` file in the experiment's processed data directory. Other attributes such as the `genes`, `tf_names`, `tg_names`, and `peaks` are loaded from the saved files in the directory. Once the TDF object is loaded, we can verify that all of the necessary files are cached properly using the `create_or_load_data_cache()` method. 

> Note: The `create_or_load_data_cache()` method is also a nice shortcut for running all of the steps to create the cached data files

In [3]:
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/")

cell_type="Macrophage"
sample_name="buffer_1"
experiment_name=f"{cell_type}_{sample_name}_tutorial"
organism_code="hg38"

tdf = data_formatter.load_tdf(
    settings_path = DATA_DIR / "PROCESSED_DATA" / experiment_name / "settings.json"
)

# Verify that the data cache files exist. If not, this method will create them.
tdf.create_or_load_data_cache(sample_name=tdf.sample_names[0], force_recalculate=False)

Macrophage_buffer_1_tutorial: Loaded existing settings
[0/4] Creating or loading pseudobulk data

[1/4] Creating or loading peak BED file

[2/4] Creating or loading peak-to-target gene distance file
Calculating peak to TSS distances for sample buffer_1 and saving to /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA/Macrophage_buffer_1_tutorial/buffer_1/peak_to_gene_dist.parquet.

[3/4] Aggregating sample-level pseudobulk datasets into global dataset
  - Loading processed pseudobulk datasets:
   - Sample names: ['buffer_1']
   - Looking for processed samples in /gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA/Macrophage_buffer_1_tutorial
  - Aggregating per-chromosome pseudobulk datasets:
   - Aggregating data for chr1
   - Aggregating data for chr2
   - Aggregating data for chr3
   - Aggregating data for chr4
   - Aggregating data for chr5
   - Aggregating data for chr6
   - Aggregating data for chr7
   - Aggregating 

## Create an ExperimentHandler

The `ExperimentHander` class uses the files and attributes from the TDF object to manage model training, gradient attribution calculations, evaluation metric calculations, and plotting. The outputs from the `ExperimentHandler` class are saved to the `experiment_dir` attribute under numbered subdirectories that separate different experiments. This allows for multiple models to be trained per experiment without overwriting the results from previous trainings.

In [39]:
importlib.reload(experiment_handler)

<module 'multiomic_transformer.utils.experiment_handler' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/experiment_handler.py'>

In [40]:
# The ExperimentHandler is a higher level class that handles the training and evaluation of the model.
# It takes in a TrainingDataFormatter object to handle file paths, data loading, and caching.
logging.info("Initializing ExperimentHandler...")
exp = experiment_handler.ExperimentHandler(
    training_data_formatter=tdf,
    experiment_dir=DATA_DIR / "EXPERIMENTS",
    model_num=1,
    silence_warnings=False,
)

Initializing ExperimentHandler...


## Create the model training loaders

Once the `ExperimentHander` is loaded, we will load in the dataset we created in Step 020 and use it to make train/val/test dataloaders for model training and evaluation.

### Creating a Multichromosome Dataset

Next we will create a `MultiChromosomeDataset` object. This is a custom PyTorch `Dataset` object that handles creating and loading batches of data that only contain TG and peak data from one chromosome at a time. This prevents the model from trying to associate peaks from one chromosome with TGs from another chromosome, and allows us to use the peak to gene distance as a bias.

> TIP: The `max_cached` argument controls how many chromosomes worth of data can be cached in memory at once. Higher values are don't require as many file I/O operations, but requires more RAM. This is a good knob to turn if you run into Out of Memory (OOM) issues.

In [41]:
logging.info("Creating dataset...")
exp.create_multichrom_dataset(max_cached=100)

Creating dataset...


### Preparing the Train/Val/Test DataLoaders

Next, we split the `MultiChromosomeDataset` object into train/val/test PyTorch `DataLoader` objects. The `prepare_dataloader()` method handles making 70% train / 15% val / 15% test splits and keeps track of which chromosome each batch came from. The method balances the splits to have an even distribution of data from each chromosome. During training and inference, each batch of data will come from each chromosome sequentially starting from chromosome 1. 

You can set the `num_workers` argument based on the number of CPU's available. These workers help load and distribute the batches of data efficiently. 

> TIP: If training is being bottlenecked by the dataloading, try increasing the `batch_size` and `num_workers` to help reduce the number of times each batch needs to be loaded.

In [42]:
# Prepares the Train/Val/Test dataloaders, being careful to balance the number of 
# batches from each chromosome in each set.
logging.info("Preparing DataLoader...")
train_loader, val_loader, test_loader = exp.prepare_dataloader(
    batch_size=64,
    num_workers=8
)

Preparing DataLoader...


### Creating Data Scalers

Once the dataset and dataloaders are created, we create a `SimpleScaler` object that z-scale normalizes the data to have a mean of 0 and a standard deviation of 1. The model is trained and makes predictions in z-space, then the data is inverse transformed to get the actual expression prediction. This helps to keep the data neat and orderly for the model. 

The mean and standard deviation is calculated using only the train split, then applied to the val and test splits. This prevents data leakage between the three splits.

In [43]:
# Creates scalers for the RNA and ATAC data based on the training split.
logging.info("Creating scalers...")
exp.create_scalers(train_loader)

Creating scalers...


## Dataset and Model Setup

### Create a new model

Next, we will create a new untrained model and update the model settings. You can check the default model settings
using the `exp.print_model_settings()` method.

In [44]:
exp.print_model_settings()

Model settings for Macrophage_buffer_1_tutorial:
  Epochs (epochs): 250
  Batch Size (batch_size): 64
  Gradient Accumulation Steps (grad_accum_steps): 1
  Use Gradient Checkpointing (use_grad_ckpt): True
  Model Dimension (d_model): 128
  Number of Heads (num_heads): 4
  Number of Layers (num_layers): 3
  Feed-Forward Dimension (d_ff): 512
  Kernel Size (kernel_size): 64
  Dropout (dropout): 0.1
  Bias Scale (bias_scale): 2.0
  Use Distance Bias (use_dist_bias): True
  Initial Learning Rate (initial_lr): 0.00025


Let's practice updating the model settings. Change the size of the model from 128 to 64. Checking the model settings again, we see that the model dimensionality has changed.

In [23]:
exp.d_model = 128
exp.print_model_settings()

Model settings for Macrophage_buffer_1_tutorial:
  Epochs (epochs): 250
  Batch Size (batch_size): 64
  Gradient Accumulation Steps (grad_accum_steps): 1
  Use Gradient Checkpointing (use_grad_ckpt): True
  Model Dimension (d_model): 128
  Number of Heads (num_heads): 4
  Number of Layers (num_layers): 3
  Feed-Forward Dimension (d_ff): 512
  Kernel Size (kernel_size): 64
  Dropout (dropout): 0.1
  Bias Scale (bias_scale): 2.0
  Use Distance Bias (use_dist_bias): True
  Initial Learning Rate (initial_lr): 0.00025


Great! We are almost to the fun part. We can create a blank `MultiomicTransformer` model using the `create_new_model()` method. You can also update the model settings here if you prefer.

> TIP: The `create_multichrom_dataset()` method needs to run before this step. The model requires the number of TFs and TGs from the dataset to set the vocab size.

In [25]:
logging.info("Creating model")
# You can leave the arguments blank to use whatever is set in the ExperimentHandler's attributes, 
# I'm just specifying them here to show you how to override them when creating the model. 
model = exp.create_new_model(
    d_model=exp.d_model,
    num_heads=exp.num_heads,
    num_layers=exp.num_layers,
    d_ff=exp.d_ff,
    dropout=exp.dropout,
    use_dist_bias=exp.use_dist_bias,
    bias_scale=exp.bias_scale,
    kernel_size=exp.kernel_size
)


Creating model


The new `MultiomicTransformer` object is stored as `exp.model`. You can print out the parameters if you want a quick look at how the architecture is set up.

In [ ]:
exp.model

Awesome! Now we are ready to move on to training. Spin up the GPU's if you got 'em.

## MultiomicTransformer model training

The model training is abstracted pretty heavily in the `train()` method. 

#### Quality of Life Parameters
- **silence_tqdm**: Display a TQDM progress bar with the number of batches completed for the epoch

- **allow_overwrite**: Allows the model training to overwrite an existing trained model in the `self.model_training_dir`. Set to `False` to create a new training directory if a `"trained_model.pt"` is found in the directory. Automatically iterates the `self.model_training_dir` to the next number.

- **verbose**: Set to `False` to silence training logs.

#### Model Learning Rate
The model uses the Adam optimizer with a `ReduceLROnPlateau` learning rate scheduler. The learning rate starts at `self.initial_lr` learning rate (default 2.5e-4). As the model learns, the learning rate will be reduced by `self.scheduler_reduction_factor` if the model doesn't improve after `self.scheduler_patience_epochs`.

#### Training Logs and Stats
Each of the logs are saved to CSV files in the model training directory.

- **GPU Memory Log**: If you are using a GPU and have `monitor_gpu_memory=True`, then the training process will track how much of the GPU is being used at each epoch. Stored as `self.gpu_mem_log_df` after model training.

- **Batch Profile Log**: This stores the time required to complete each step of the training (loading the data, transfering the data to the GPU, scaling the data, running the forward and backward passes, and the total epoch time). It also stores information about the batch size and the number of ATAC windows, TFs, and TGs. This is useful for identifying any bottlenecks in the training. Stored as `self.batch_profile_df` after model training.

- **Epoch Log**: This stores the logging information displayed during training, including the epoch, loss, $R^{2}$ correlation, time per epoch, and last epoch with improvement. Stored as `self.epoch_log_df`.

In [26]:

# Runs model training and returns the trained model.
logging.info("Training model")
model = exp.train(
    train_loader=train_loader, 
    val_loader=val_loader, 
    num_epochs=200,
    learning_rate=exp.initial_lr,
    max_batches=None,
    verbose=True,
    save_every_n_epochs=10,
    monitor_gpu_memory=True,
    profile_batches=True,
    allow_overwrite=True,
    silence_tqdm=True,
    )


Training model

===== Model 1 Training Started =====
Epoch 0/200 | Train Loss: 1.0563 | Val MSE: 0.0862 | R2 (Unscaled): -0.005 | R2 (Scaled): -0.008 | LR: 2.50e-04 | Time: 12.0s | Last Improved: 0 epochs ago | Checkpoint Saved
Epoch 1/200 | Train Loss: 1.0113 | Val MSE: 0.0860 | R2 (Unscaled): -0.002 | R2 (Scaled): -0.005 | LR: 2.50e-04 | Time: 4.3s | Last Improved: 0 epochs ago | 
Epoch 2/200 | Train Loss: 1.0076 | Val MSE: 0.0859 | R2 (Unscaled): -0.001 | R2 (Scaled): -0.004 | LR: 2.50e-04 | Time: 4.4s | Last Improved: 0 epochs ago | 
Epoch 3/200 | Train Loss: 1.0042 | Val MSE: 0.0857 | R2 (Unscaled): 0.000 | R2 (Scaled): -0.002 | LR: 2.50e-04 | Time: 4.3s | Last Improved: 0 epochs ago | 
Epoch 4/200 | Train Loss: 0.9995 | Val MSE: 0.0855 | R2 (Unscaled): 0.002 | R2 (Scaled): -0.001 | LR: 2.50e-04 | Time: 4.4s | Last Improved: 0 epochs ago | 
Epoch 5/200 | Train Loss: 0.9952 | Val MSE: 0.0853 | R2 (Unscaled): 0.005 | R2 (Scaled): 0.001 | LR: 2.50e-04 | Time: 4.3s | Last Improved: 0 

### Gradient Attribution Calculation

Once the model has finished training, we will generate the TF-TG GRN using the gradient attribution method. This averages the per-batch loss gradient between the TF input and the TG expression output to get an approximation of how important each TF is to the TG expression prediction for each batch.

The final GRN is stored as a pandas DataFrame to `self.grn` with columns `Source`, `Target`, and `Score`. The GRN is also saved in `self.model_training_dir` as `"inferred_grn.csv"`.

> TIP: To reload a saved GRN, run the `exp.load_grn()` method. This will look for the `"inferred_grn.csv"` file in the `self.model_training_dir` and load it to `exp.grn`.

In [45]:
model = exp.load_model()

In [46]:
# Runs gradient attribution to calculate the gradients between each TF input and each TG output.
logging.info("\nRunning Gradient Attribution")
exp.run_atac_gradient_attribution(
    test_loader,
    chunk_size=16,
    max_batches=None,
    show_tqdm=True,
    )


Running Gradient Attribution


13366


ATAC Gradient attributions:  98%|██████████████████████████████████▏| 43/44 [02:35<00:03,  3.63s/it]


(                           Source   Target     Score
 0              CHR1:923373-924275  A3GALT2  0.354727
 1              CHR1:923373-924275   ABCB10 -0.252396
 2              CHR1:923373-924275    ABCD3 -0.510703
 3              CHR1:923373-924275     ABL2  1.044350
 4              CHR1:923373-924275    ACADM  0.538197
 ...                           ...      ...       ...
 43112109  CHR22:50785018-50785904    YWHAH  0.000000
 43112110  CHR22:50785018-50785904   ZC3H7B  0.000000
 43112111  CHR22:50785018-50785904   ZDHHC8  0.000000
 43112112  CHR22:50785018-50785904    ZMAT5  0.000000
 43112113  CHR22:50785018-50785904    ZNRF3  0.000000
 
 [43112114 rows x 3 columns],
 {0:                            Source   Target     Score
  0              chr1:923373-924275  A3GALT2  0.002287
  1              chr1:923373-924275   ABCB10  0.001590
  2              chr1:923373-924275    ABCD3  0.001573
  3              chr1:923373-924275     ABL2  0.003500
  4              chr1:923373-924275    ACA

In [38]:

# Runs gradient attribution to calculate the gradients between each TF input and each TG output.
logging.info("\nRunning Gradient Attribution")
exp.run_gradient_attribution(
    test_loader,
    chunk_size=64,
    max_batches=None,
    max_tgs_per_batch=None,
    show_tqdm=True,
    )


Running Gradient Attribution
Gradient attributions: 100%|███████████████████████████████████| 44/44 [02:19<00:00,  3.16s/batches]


(            Source   Target     Score
 0         AC092835     A1BG  0.379014
 1         AC092835      A2M  0.389120
 2         AC092835    A2ML1 -0.737082
 3         AC092835  A3GALT2 -0.694372
 4         AC092835   A4GALT -0.571215
 ...            ...      ...       ...
 14770168      ZZZ3   ZWILCH  0.483171
 14770169      ZZZ3    ZWINT  0.375368
 14770170      ZZZ3   ZYG11B  0.536402
 14770171      ZZZ3      ZYX  1.553063
 14770172      ZZZ3    ZZEF1  0.770117
 
 [14770173 rows x 3 columns],
 {0:            Source   Target     Score
  0        AC092835  A3GALT2  0.000014
  1        AC092835   ABCB10  0.000036
  2        AC092835    ABCD3  0.000033
  3        AC092835     ABL2  0.000057
  4        AC092835    ACADM  0.000036
  ...           ...      ...       ...
  1600715      ZZZ3   ZNF593  0.000895
  1600716      ZZZ3   ZNHIT6  0.001186
  1600717      ZZZ3   ZRANB2  0.002456
  1600718      ZZZ3   ZSWIM5  0.001310
  1600719      ZZZ3   ZYG11B  0.002040
  
  [1600720 rows x 3 column

In [ ]:
# Loads a ground truth file with columns "Source" and "Target" for TF-TG interactions.
logging.info("Loading ground truth datasets...")
GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
gt_by_dataset_dict = {
    "ChIP-Atlas macrophage": exp.load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_macrophage.csv"),
    "Perturb-seq macrophage": exp.load_ground_truth(GROUND_TRUTH_DIR / "macrophage_perturbations.csv"),
}

# Calculates the AUROC of the predicted GRN against multiple ground truth datasets.
logging.info("\nCalculating AUROC")
auroc_df = exp.calculate_auroc_all_sample_gts(exp.grn, gt_by_dataset_dict)     
logging.info(f"Pooled Median AUROC: {auroc_df['pooled_median_auroc'].iloc[0]:.3f}")       
logging.info(f"Per-TF Median AUROC: {auroc_df['per_tf_median_auroc'].iloc[0]:.3f}")

exp.save_handler()

In [ ]:
from multiomic_transformer.utils import auroc_refactored
importlib.reload(auroc_refactored)

In [ ]:
linger_grn_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/other_method_grns/LINGER_muon")
scenic_grn_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/other_method_grns/SCENIC_muon")

def print_grn_stats(grn_dir, method):
    logging.info(f"\n{method} GRN stats:")
    df_dict = {
        "method": [],
        "sample_name": [],
        "tfs": [],
        "tgs": [],
        "edges": [],
    }
    
    for file in grn_dir.glob("*.tsv"):
        method_name = file.stem
        method_name = method_name.replace(method, "")
        cell_type = method_name.split("_")[0]
        sample_name = "_".join(method_name.split("_")[1:])
        
        grn_df = pd.read_csv(file, sep="\t")
        n_tfs = grn_df["Source"].nunique()
        n_tgs = grn_df["Target"].nunique()
        n_edges = len(grn_df)
        logging.info(f"  - {cell_type} {sample_name}: {n_tfs:,} TFs, {n_tgs:,} TGs, {n_edges:,} edges")
        df_dict["method"].append(method)
        df_dict["sample_name"].append(sample_name)
        df_dict["tfs"].append(n_tfs)
        df_dict["tgs"].append(n_tgs)
        df_dict["edges"].append(n_edges)
        
    return pd.DataFrame(df_dict)

scenic_grn_size_df = print_grn_stats(scenic_grn_dir, "scenic_plus")
linger_grn_size_df = print_grn_stats(linger_grn_dir, "linger")

combined_size_df = pd.concat([scenic_grn_size_df, linger_grn_size_df], ignore_index=True)

In [ ]:
sample_order = [
    "mESC_E7.5_rep1",
    "mESC_E7.5_rep2",
    "mESC_E8.5_rep1",
    "mESC_E8.5_rep2",
    "Macrophage_buffer_1",
    "Macrophage_buffer_2",
    "Macrophage_buffer_3",
    "Macrophage_buffer_4",
    "K562_sample_1",
    "iPSC_WT_D13_rep1"
]

linger_grn_size_df = linger_grn_size_df.sort_values(by="sample_name", key=lambda x: x.map({name: i for i, name in enumerate(sample_order)}))
scenic_grn_size_df = scenic_grn_size_df.sort_values(by="sample_name", key=lambda x: x.map({name: i for i, name in enumerate(sample_order)}))

combined_size_df = pd.concat([scenic_grn_size_df, linger_grn_size_df], ignore_index=True)
combined_size_df

In [ ]:
linger_grn_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/other_method_grns/LINGER_muon")

for grn_file in linger_grn_dir.glob("*.tsv"):
    logging.info(f"Loading LINGER GRN file: {grn_file.name}")

    df = pd.read_csv(grn_file, sep="\t", index_col=0).reset_index()

    cols = set(df.columns)

    # Case 1: already long
    if {"Source", "Target", "Score"}.issubset(cols):
        
        if df["Source"].nunique() > df["Target"].nunique():
            logging.info(f"  - Renaming columns for {grn_file.name}")
            df = df.rename(columns={"Source": "Target", "Target": "Source"})
            df = df[["Source", "Target", "Score"]]
            df.to_csv(grn_file, sep="\t", index=False)
        else:
            logging.info(f"  - {grn_file.name} already appears to be in long format. Skipping.")
        
        continue

    # Case 2: likely wide
    source_col = df.columns[0]

    # Heuristic: wide GRN matrix usually has many columns beyond the first identifier column
    if len(df.columns) > 3:
        df_long = (
            df.melt(
                id_vars=source_col,
                var_name="Target",
                value_name="Score"
            )
            .rename(columns={source_col: "Source"})
        )

        # Optional: remove missing values
        df_long = df_long.dropna(subset=["Score"])

        out_file = grn_file.with_name(grn_file.stem + "_long.tsv")
        df_long.to_csv(out_file, sep="\t", index=False)
        logging.info(f"  - Saved melted GRN to {out_file.name}")
    else:
        logging.warning(f"  - Could not confidently determine format for {grn_file.name}. Skipping.")

## Loading a saved ExperimentHandler

In [ ]:
importlib.reload(experiment_handler)
importlib.reload(data_formatter)

In [ ]:
def combine_images_with_spans(
    layout,
    space,
    fig_dir,
    output_name="combined_images.png",
    bg_color=(255, 255, 255, 255),
):
    """
    Combine images into a manually positioned grid with optional row/column spanning.

    Parameters
    ----------
    layout : dict
        Dictionary mapping image path -> config dict

        Required keys per image:
            row : int
            col : int

        Optional keys per image:
            scale : float, default 1.0
            rowspan : int, default 1
            colspan : int, default 1

        Example:
        {
            Path("a.png"): {"row": 0, "col": 0, "scale": 1.0, "rowspan": 1, "colspan": 2},
            Path("b.png"): {"row": 0, "col": 2, "scale": 0.8},
            Path("c.png"): {"row": 1, "col": 0, "scale": 1.1, "rowspan": 2, "colspan": 2},
        }

    space : int
        Pixels between grid cells.
    fig_dir : Path
        Output directory.
    output_name : str
        Output filename.
    bg_color : tuple
        RGBA background color.

    Returns
    -------
    Path
        Path to the saved combined image.
    """
    items = []

    # Load images and parse layout
    for image_path, cfg in layout.items():
        row = cfg["row"]
        col = cfg["col"]
        scale = cfg.get("scale", 1.0)
        rowspan = cfg.get("rowspan", 1)
        colspan = cfg.get("colspan", 1)

        if rowspan < 1 or colspan < 1:
            raise ValueError(f"{image_path}: rowspan and colspan must be >= 1")

        img = PILImage.open(image_path).convert("RGBA")
        new_width = max(1, int(img.width * scale))
        new_height = max(1, int(img.height * scale))
        img_resized = img.resize((new_width, new_height), PILImage.LANCZOS)

        items.append({
            "path": image_path,
            "row": row,
            "col": col,
            "rowspan": rowspan,
            "colspan": colspan,
            "img": img_resized,
            "width": new_width,
            "height": new_height,
        })

    if not items:
        raise ValueError("Layout is empty.")

    n_rows = max(item["row"] + item["rowspan"] for item in items)
    n_cols = max(item["col"] + item["colspan"] for item in items)

    # Check for overlapping occupied cells
    occupied = {}
    for item in items:
        for r in range(item["row"], item["row"] + item["rowspan"]):
            for c in range(item["col"], item["col"] + item["colspan"]):
                key = (r, c)
                if key in occupied:
                    raise ValueError(
                        f"Overlap detected: {item['path']} overlaps with {occupied[key]} at cell {key}"
                    )
                occupied[key] = item["path"]

    # Compute minimum column widths and row heights
    # Start from single-cell requirements
    col_widths = [0] * n_cols
    row_heights = [0] * n_rows

    for item in items:
        if item["colspan"] == 1:
            col_widths[item["col"]] = max(col_widths[item["col"]], item["width"])
        if item["rowspan"] == 1:
            row_heights[item["row"]] = max(row_heights[item["row"]], item["height"])

    # Expand widths/heights as needed for spanned items
    # A simple iterative pass is usually enough for plotting layouts
    for _ in range(10):
        changed = False

        for item in items:
            # Width across spanned columns, including inter-column spaces inside the span
            current_span_width = (
                sum(col_widths[item["col"]: item["col"] + item["colspan"]])
                + space * (item["colspan"] - 1)
            )
            if current_span_width < item["width"]:
                deficit = item["width"] - current_span_width
                add_per_col = deficit / item["colspan"]
                for c in range(item["col"], item["col"] + item["colspan"]):
                    new_val = int(round(col_widths[c] + add_per_col))
                    if new_val > col_widths[c]:
                        col_widths[c] = new_val
                        changed = True

            # Height across spanned rows, including inter-row spaces inside the span
            current_span_height = (
                sum(row_heights[item["row"]: item["row"] + item["rowspan"]])
                + space * (item["rowspan"] - 1)
            )
            if current_span_height < item["height"]:
                deficit = item["height"] - current_span_height
                add_per_row = deficit / item["rowspan"]
                for r in range(item["row"], item["row"] + item["rowspan"]):
                    new_val = int(round(row_heights[r] + add_per_row))
                    if new_val > row_heights[r]:
                        row_heights[r] = new_val
                        changed = True

        if not changed:
            break

    # Compute canvas size
    background_width = sum(col_widths) + space * (n_cols - 1)
    background_height = sum(row_heights) + space * (n_rows - 1)

    background = PILImage.new("RGBA", (background_width, background_height), bg_color)

    # Precompute grid start coordinates
    col_starts = []
    x = 0
    for w in col_widths:
        col_starts.append(x)
        x += w + space

    row_starts = []
    y = 0
    for h in row_heights:
        row_starts.append(y)
        y += h + space

    # Paste each image centered within its spanned rectangle
    for item in items:
        x0 = col_starts[item["col"]]
        y0 = row_starts[item["row"]]

        span_width = (
            sum(col_widths[item["col"]: item["col"] + item["colspan"]])
            + space * (item["colspan"] - 1)
        )
        span_height = (
            sum(row_heights[item["row"]: item["row"] + item["rowspan"]])
            + space * (item["rowspan"] - 1)
        )

        x_offset = (span_width - item["width"]) // 2
        y_offset = (span_height - item["height"]) // 2

        paste_x = x0 + x_offset
        paste_y = y0 + y_offset

        background.paste(item["img"], (paste_x, paste_y), item["img"])

    output_path = fig_dir / output_name
    background.save(output_path)
    return output_path

def generate_all_plots(exp: experiment_handler.ExperimentHandler, gt_by_dataset_dict_sample: dict):
    logging.info(f"\n----- GRN Overlap with Ground Truth -----")
    for ground_truth_name, ground_truth in gt_by_dataset_dict_sample.items():
        exp.report_grn_overlap_with_gt(ground_truth_name, ground_truth)

    logging.info(f"\n----- AUROC -----")
    per_tf_df_all =  pd.read_csv(exp.model_training_dir / "per_tf_auroc_auprc_results.csv")
    pooled_df_all = pd.read_csv(exp.model_training_dir / "pooled_auroc_auprc_results.csv")

    per_tf_plot_df = (
        per_tf_df_all.dropna(subset=["auroc"])
        .groupby(['method', 'gt'], as_index=False)
        .agg(
            auroc=('auroc', 'median'),
        )
    )

    pooled_df_median_auroc = pooled_df_all[pooled_df_all["method"] == "Gradient Attribution"]["auroc"].median()
    per_tf_median_auroc = per_tf_plot_df[per_tf_plot_df["method"] == "Gradient Attribution"]["auroc"].median()

    logging.info(f"Median Pooled AUROC: {pooled_df_median_auroc:.3f}")
    logging.info(f"Median Per-TF AUROC: {per_tf_median_auroc:.3f}")

    fig_dir = exp.model_training_dir / "figures"
    fig_dir.mkdir(exist_ok=True)

    ground_truth_name = list(gt_by_dataset_dict_sample.keys())[0]

    name = exp.experiment_name.replace("_", " ")
    per_tf_plot, per_tf_df, tf_curves = exp.plot_top_n_tf_roc_curves(
        exp.grn, 
        gt_by_dataset_dict_sample[ground_truth_name], 
        ground_truth_name, 
        exp, 
        method_name="Gradient Attribution", 
        num_top_tfs_to_plot=10,
        min_edges=500,
        min_pos=50,
        balance=True,
        name_clean=name,
        override_title=f"{ground_truth_name}\nTop 10 Per-TF AUROC"
        )
    per_tf_plot.savefig(fig_dir / f"top_10_per_tf_auroc.png", dpi=250, bbox_inches="tight")
    plt.close(per_tf_plot)

    pooled_auroc_boxplot_fig = exp._plot_all_results_auroc_boxplot(
        pooled_df_all,
        per_tf=False,
        ylim=(0.2, 0.8),
        override_title=f"Pooled AUROC per Method",
        method_color_dict=exp.method_color_dict
    )
    pooled_auroc_boxplot_fig.savefig(fig_dir / f"pooled_auroc_boxplot.png", dpi=250, bbox_inches="tight")
    plt.close(pooled_auroc_boxplot_fig)

    per_tf_auroc_boxplot_fig = exp._plot_all_results_auroc_boxplot(
        per_tf_plot_df, 
        per_tf=True,
        ylim=(0.2, 0.8),
        override_title=f"Per-TF AUROC per Method",
        method_color_dict=exp.method_color_dict
        )
    per_tf_auroc_boxplot_fig.savefig(fig_dir / f"per_tf_auroc_boxplot.png", dpi=250, bbox_inches="tight")
    plt.close(per_tf_auroc_boxplot_fig)

    relative_improvement_fig = exp.plot_relative_improvement(
        per_tf_plot_df, 
        exp.experiment_name,
        override_title=f"Per-TF AUROC Improvement",
        )
    relative_improvement_fig.savefig(fig_dir / f"relative_improvement.png", dpi=250, bbox_inches="tight")
    plt.close(relative_improvement_fig)

    pooled_auroc_heatmap_fig = exp.plot_method_gt_heatmap(
        pooled_df_all, 
        per_tf=False,
        x_scale=1.2,
        y_scale=0.6,
        override_title=f"Pooled AUROC by Method and GT"
        )
    pooled_auroc_heatmap_fig.savefig(fig_dir / f"pooled_auroc_heatmap.png", dpi=250, bbox_inches="tight")
    plt.close(pooled_auroc_heatmap_fig)

    per_tf_auroc_heatmap_fig = exp.plot_method_gt_heatmap(
        per_tf_plot_df, 
        per_tf=True,
        x_scale=1.2,
        y_scale=0.6,
        override_title=f"Per-TF AUROC by Method and GT"
        )
    per_tf_auroc_heatmap_fig.savefig(fig_dir / f"per_tf_auroc_heatmap.png", dpi=250, bbox_inches="tight")
    plt.close(per_tf_auroc_heatmap_fig)

    true_vs_predicted_fig = exp.plot_true_vs_predicted_tg_expression(
        num_batches=len(exp.test_loader), 
        set_axis_logscale=False,
        title=f"Predicted vs True TG Expression"
        )
    true_vs_predicted_fig.savefig(fig_dir / f"true_vs_predicted.png", dpi=250, bbox_inches="tight")
    plt.close(true_vs_predicted_fig)
    
    layout = {
        fig_dir / "top_10_per_tf_auroc.png": {
            "row": 0, "col": 0, "scale": 0.9, "rowspan": 1, "colspan": 2
        },
        fig_dir / "pooled_auroc_boxplot.png": {
            "row": 0, "col": 2, "scale": 0.9
        },
        fig_dir / "pooled_auroc_heatmap.png": {
            "row": 0, "col": 3, "scale": 1.0
        },
        fig_dir / "true_vs_predicted.png": {
            "row": 1, "col": 0, "scale": 0.85
        },
        fig_dir / "relative_improvement.png": {
            "row": 1, "col": 1, "scale": 0.85
        },
        fig_dir / "per_tf_auroc_boxplot.png": {
            "row": 1, "col": 2, "scale": 0.9
        },
        fig_dir / "per_tf_auroc_heatmap.png": {
            "row": 1, "col": 3, "scale": 1.0
        },
    }

    output_path = combine_images_with_spans(
        layout=layout,
        space=50,
        fig_dir=fig_dir,
        output_name="combined_images.png",
    )
    
    return output_path

def rank_methods_by_auroc(df):
    """Calculates the sorted rank by median AUROC for each inference method"""
    df_grouped = (
        df
        .groupby("method", as_index=False)
        .agg(median_auroc=("auroc", "median"))
        .sort_values(by="median_auroc", ascending=False)
        .reset_index(drop=True)
        .reset_index(drop=False)
        .rename(columns={"index": "rank"})
        .assign(rank=lambda x: x["rank"] + 1)
    )
    return df_grouped[["method", "median_auroc", "rank"]]

def get_method_rank(df, method_name):
    """Calculates the sorted rank by median AUROC for an inference method"""
    df_grouped = rank_methods_by_auroc(df)
    method_rank = df_grouped[df_grouped["method"] == method_name]["rank"].values[0]
    return method_rank 

def avg_rank_by_method_plot(avg_rank_df, method_color_dict, title, rename_map=None):
    fig = plt.figure(figsize=(6, 4))

    order = avg_rank_df["method"].tolist()

    sns.barplot(
        data=avg_rank_df,
        x="method",
        y="avg_rank",
        order=order,
        palette=method_color_dict
    )

    ax = plt.gca()
    
    if rename_map is None:
        rename_map = {}

    # Rename + recolor ticks
    new_labels = []
    for tick in ax.get_xticklabels():
        original = tick.get_text()
        new = rename_map.get(original, original)
        new_labels.append(new)

        # Only color MTGRN
        if new == "MTGRN":
            tick.set_color(method_color_dict.get(original, "black"))
            tick.set_fontweight("bold")
        else:
            tick.set_color("black")
            tick.set_fontweight("normal")

    ax.set_xticklabels(new_labels, rotation=45, ha="right")

    plt.ylabel("Average Rank")
    plt.xlabel("")
    plt.title(title)
    plt.tight_layout()
    return fig

def experiment_by_method_rank_heatmap(all_ranks_df, rank_df, method_color_dict, experiment_order, rename_map=None):
    if rename_map is None:
        rename_map = {}

    rank_heatmap_df = all_ranks_df.pivot(
        index="experiment",
        columns="method",
        values="rank"
    )

    method_order = rank_df["method"].tolist()

    rank_heatmap_df = rank_heatmap_df.reindex(
        index=experiment_order,
        columns=method_order
    )

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        rank_heatmap_df,
        annot=True,
        fmt=".0f",
        cmap="viridis_r",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Rank"},
        ax=ax
    )

    # Rename + recolor ticks
    new_labels = []
    for tick in ax.get_xticklabels():
        original = tick.get_text()
        new = rename_map.get(original, original)
        new_labels.append(new)

        if new == "MTGRN":
            tick.set_color(method_color_dict.get(original, "black"))
            tick.set_fontweight("bold")
        else:
            tick.set_color("black")
            tick.set_fontweight("normal")

    ax.set_xticklabels(new_labels, rotation=45, ha="right")

    ax.set_title("Method Rank by Experiment")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()

    return fig

def per_tf_method_rank_plots(selected_experiments, method_color_dict, rename_map=None):

    all_ranks = []

    for sample_name, experiment_name, sample_type, model_num in selected_experiments:

        exp = experiment_handler.load_experiment_handler(
            tdf_settings_path=DATA_DIR / experiment_name / "settings.json",
            experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/"),
            model_num=model_num,
        )
        if not (exp.model_training_dir / "per_tf_auroc_auprc_results.csv").exists():
            logging.warning(f"{exp.experiment_name} does not have per-TF results. Skipping.")
            continue
        
        per_tf_df_all =  pd.read_csv(exp.model_training_dir / "per_tf_auroc_auprc_results.csv")

        per_tf_df_all = per_tf_df_all[per_tf_df_all["method"] != "TF Knockout"]

        per_tf_plot_df = (
            per_tf_df_all.dropna(subset=["auroc"])
            .groupby(['method', 'gt'], as_index=False)
            .agg(
                auroc=('auroc', 'median'),
            )
        )
        
        per_tf_df_grouped = (
            per_tf_plot_df
            .groupby("method", as_index=False)
            .agg(median_auroc=("auroc", "median"))
            .sort_values(by="median_auroc", ascending=False)
            .reset_index(drop=True)
            .reset_index(drop=False)
            .rename(columns={"index": "rank"})
            .assign(rank=lambda x: x["rank"] + 1)
        )

        # Add experiment identifier
        per_tf_df_grouped["experiment"] = sample_name

        all_ranks.append(per_tf_df_grouped[["experiment", "method", "rank"]])
        
    all_ranks_df = pd.concat(all_ranks, ignore_index=True)

    final_ranking = (
        all_ranks_df
        .groupby("method", as_index=False)
        .agg(
            total_rank=("rank", "sum"),
            avg_rank=("rank", "mean"),
            n_experiments=("rank", "count")
        )
        .sort_values(by="total_rank")  # lower = better
        .reset_index(drop=True)
    )

    # display(final_ranking)

    fig = avg_rank_by_method_plot(final_ranking, method_color_dict, title="Average Method Ranking by Per-TF AUROC", rename_map=rename_map)
    fig.show()

    experiment_order = [sample_name for sample_name, _, _, _ in selected_experiments]
    fig = experiment_by_method_rank_heatmap(all_ranks_df, final_ranking, method_color_dict, experiment_order=experiment_order, rename_map=rename_map)
    fig.show()
    
def pooled_method_rank_plots(selected_experiments, method_color_dict, rename_map=None):
    all_ranks = []

    for sample_name, experiment_name, sample_type, model_num in selected_experiments:

        exp = experiment_handler.load_experiment_handler(
            tdf_settings_path=DATA_DIR / experiment_name / "settings.json",
            experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/"),
            model_num=model_num,
        )
        if not (exp.model_training_dir / "pooled_auroc_auprc_results.csv").exists():
            logging.warning(f"{exp.experiment_name} does not have pooled results. Skipping.")
            continue

        pooled_df_all = pd.read_csv(exp.model_training_dir / "pooled_auroc_auprc_results.csv")

        pooled_df_all = pooled_df_all[pooled_df_all["method"] != "TF Knockout"]

        pooled_df_grouped = rank_methods_by_auroc(pooled_df_all)

        # Add experiment identifier
        pooled_df_grouped["experiment"] = sample_name

        all_ranks.append(pooled_df_grouped[["experiment", "method", "rank"]])
        
    all_ranks_df = pd.concat(all_ranks, ignore_index=True)

    final_ranking = (
        all_ranks_df
        .groupby("method", as_index=False)
        .agg(
            total_rank=("rank", "sum"),
            avg_rank=("rank", "mean"),
            n_experiments=("rank", "count")
        )
        .sort_values(by="total_rank")  # lower = better
        .reset_index(drop=True)
    )

    # display(final_ranking)

    fig = avg_rank_by_method_plot(final_ranking, method_color_dict, title="Average Method Ranking by Pooled AUROC", rename_map=rename_map)
    fig.show()

    experiment_order = [sample_name for sample_name, _, _, _ in selected_experiments]
    fig = experiment_by_method_rank_heatmap(all_ranks_df, final_ranking, method_color_dict, experiment_order=experiment_order, rename_map=rename_map)
    fig.show()

In [ ]:
def load_ground_truth(ground_truth_file: str|Path):
    if type(ground_truth_file) == str:
        ground_truth_file = Path(ground_truth_file)
        
    if ground_truth_file.suffix == ".csv":
        sep = ","
    elif ground_truth_file.suffix == ".tsv":
        sep="\t"
        
    ground_truth_df = pd.read_csv(ground_truth_file, sep=sep, on_bad_lines="skip", engine="python")
    
    if "chip" in ground_truth_file.name and "atlas" in ground_truth_file.name:
        ground_truth_df = ground_truth_df[["source_id", "target_id"]]

    if ground_truth_df.columns[0] != "Source" or ground_truth_df.columns[1] != "Target":
        ground_truth_df = ground_truth_df.rename(columns={ground_truth_df.columns[0]: "Source", ground_truth_df.columns[1]: "Target"})
    ground_truth_df["Source"] = ground_truth_df["Source"].astype(str).str.upper()
    ground_truth_df["Target"] = ground_truth_df["Target"].astype(str).str.upper()
    
    # Build TF, TG, and edge sets for quick lookup later
    gt = ground_truth_df[["Source", "Target"]].dropna()

    gt_tfs = set(gt["Source"].unique())
    gt_tgs = set(gt["Target"].unique())
    
    gt_pairs = (gt["Source"] + "\t" + gt["Target"]).drop_duplicates()
    
    gt_lookup = (gt_tfs, gt_tgs, set(gt_pairs))
        
    return ground_truth_df, gt_lookup

gt_by_dataset_dict = {
    "Macrophage": {
        # "RN204": load_ground_truth(GROUND_TRUTH_DIR / "rn204_macrophage_human_chipseq.tsv"),
        "ChIP-Atlas macrophage": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_macrophage.csv"),
        "Perturb-seq macrophage": load_ground_truth(GROUND_TRUTH_DIR / "macrophage_perturbations.csv"),
    },
    "mESC": {
        "ChIP-Atlas mESC": load_ground_truth(GROUND_TRUTH_DIR / "chip_atlas_tf_peak_tg_dist.csv"),
        "RN111": load_ground_truth(GROUND_TRUTH_DIR / "RN111.tsv"),
        "RN112": load_ground_truth(GROUND_TRUTH_DIR / "RN112.tsv"),
        "RN114": load_ground_truth(GROUND_TRUTH_DIR / "RN114.tsv"),
        "RN116": load_ground_truth(GROUND_TRUTH_DIR / "RN116.tsv"),        
    },
    "K562": {
        "ChIP-Atlas K562": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_K562.csv"),
        "RN117": load_ground_truth(GROUND_TRUTH_DIR / "RN117.tsv"),        
    },
    "iPSC": {
        # "ChIP-Atlas iPSC": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC.csv"),
        "ChIP-Atlas iPSC (1 Mb)": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC_1mb.csv"),
        # "ChIP-Atlas iPSC (100 kb)": load_ground_truth(GROUND_TRUTH_DIR / "chipatlas_iPSC_100kb.csv"),
    }
}

In [ ]:
experiment_name = "Macrophage_buffer_1_full_pipeline_copy"
sample_type = "Macrophage"


exp = experiment_handler.load_experiment_handler(
    tdf_settings_path=DATA_DIR / experiment_name / "settings.json",
    experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/"),
    model_num=2,
    )

gt_by_dataset_dict_sample = gt_by_dataset_dict[sample_type]

# output_path = generate_all_plots(exp, gt_by_dataset_dict_sample)
# Image(str(output_path))

In [ ]:
exp.calculate_auroc_all_methods(
    exp.tdf.sample_names, 
    "Macrophage", 
    gt_by_dataset_dict_sample, 
    grn=exp.grn,
    use_muon_grn=True
)

In [ ]:
fig = exp.plot_hist_roc_pr(
    exp.grn, 
    gt_by_dataset_dict_sample, 
    method_name="Gradient Attribution", 
)

In [ ]:
fig, _, _ = exp.plot_top_n_tf_roc_curves(
    exp.grn, 
    gt_by_dataset_dict_sample["Perturb-seq macrophage"], 
    "Perturb-seq macrophage", 
    exp, 
    method_name="Gradient Attribution", 
    num_top_tfs_to_plot=10,
    min_edges=10,
    min_pos=5,
    balance=True,
    name_clean=exp.experiment_name.replace("_", " "),
    override_title=f"Perturb-seq Macrophage\nTop 10 Per-TF AUROC"
)
fig.show()

### Plots for ranking all methods

In [ ]:
selected_preprocessing_experiments = [
    # ("mESC E7.5_rep1", "mESC_E7.5_rep1_full_pipeline", "mESC", 2),
    # ("mESC E7.5_rep2", "mESC_E7.5_rep2_full_pipeline", "mESC", 2),
    # ("mESC E8.5_rep1", "mESC_E8.5_rep1_full_pipeline", "mESC", 2),
    # ("mESC E8.5_rep2", "mESC_E8.5_rep2_full_pipeline", "mESC", 2),
    ("Macrophage_buffer_1", "Macrophage_buffer_1_full_pipeline_copy", "Macrophage", 2),
    # ("Macrophage_buffer_2", "Macrophage_buffer_2_full_pipeline", "Macrophage", 2),
    # ("K562", "K562_sample_1_full_pipeline", "K562", 2),
    # ("iPSC", "iPSC_WT_D13_rep1_full_pipeline", "iPSC", 2)
    ]

# rename_map = {"Gradient Attribution": "MTGRN"}
rename_map = None
    
pooled_method_rank_plots(selected_preprocessing_experiments, method_color_dict, rename_map=rename_map)
per_tf_method_rank_plots(selected_preprocessing_experiments, method_color_dict, rename_map=rename_map)

In [ ]:
importlib.reload(experiment_handler)

In [ ]:
from joblib import Parallel, delayed
from pathlib import Path
import logging
import pandas as pd
import multiomic_transformer.utils.auroc_refactored as auroc_utils


selected_preprocessing_experiments = [
    ("E7.5_rep1", "mESC_E7.5_rep1_full_pipeline", "mESC", 2),
    ("E7.5_rep2", "mESC_E7.5_rep2_full_pipeline", "mESC", 2),
    ("E8.5_rep1", "mESC_E8.5_rep1_full_pipeline", "mESC", 2),
    ("E8.5_rep2", "mESC_E8.5_rep2_full_pipeline", "mESC", 2),
    ("buffer_1", "Macrophage_buffer_1_full_pipeline", "Macrophage", 2),
    ("buffer_2", "Macrophage_buffer_2_full_pipeline", "Macrophage", 2),
    ("sample_1", "K562_sample_1_full_pipeline", "K562", 2),
    ("WT_D13_rep1", "iPSC_WT_D13_rep1_full_pipeline", "iPSC", 2),
]


def evaluate_single_method_setwise(
    method_name: str,
    score_df: pd.DataFrame,
    ground_truth: tuple[pd.DataFrame, tuple[set[str], set[str], set[str]]],
    ground_truth_name: str,
    max_edges: int = 10000,
) -> pd.DataFrame | None:
    _, gt_lookup = ground_truth
    gt_tfs, gt_targets, gt_pairs = gt_lookup

    if score_df is None or score_df.empty:
        return None

    if "source_upper" in score_df.columns and "target_upper" in score_df.columns:
        df = score_df
    else:
        df = score_df[["Source", "Target"] + (["Score"] if "Score" in score_df.columns else [])].copy()
        df["source_upper"] = df["Source"].astype(str).str.upper()
        df["target_upper"] = df["Target"].astype(str).str.upper()

    mask = df["source_upper"].isin(gt_tfs) & df["target_upper"].isin(gt_targets)
    df_filtered = df.loc[mask]

    if df_filtered.empty:
        return None

    if "Score" in df_filtered.columns:
        df_final = df_filtered.nlargest(max_edges, columns="Score", keep="first")
    else:
        df_final = df_filtered.head(max_edges)

    if df_final.empty:
        return None

    inferred_pairs = set(df_final["source_upper"] + "\t" + df_final["target_upper"])

    tp_n = len(inferred_pairs & gt_pairs)
    fp_n = len(inferred_pairs - gt_pairs)
    fn_n = len(gt_pairs - inferred_pairs)

    precision = tp_n / (tp_n + fp_n) if (tp_n + fp_n) else 0.0
    recall = tp_n / (tp_n + fn_n) if (tp_n + fn_n) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    inferred_tfs = set(df_final["source_upper"].unique())
    inferred_targets = set(df_final["target_upper"].unique())

    return pd.DataFrame([{
        "Method": method_name,
        "GT_name": ground_truth_name,
        "GT_total_edges": len(gt_pairs),
        "GT_TFs": len(gt_tfs),
        "GT_targets": len(gt_targets),
        "Original_network_size": len(score_df),
        "Filtered_network_size": len(df_filtered),
        "Final_network_size": len(inferred_pairs),
        "Inferred_TFs": len(inferred_tfs),
        "Inferred_targets": len(inferred_targets),
        "TP": tp_n,
        "FP": fp_n,
        "FN": fn_n,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1_score": round(f1, 4),
        "Common_TFs": len(gt_tfs & inferred_tfs),
        "Common_targets": len(gt_targets & inferred_targets),
    }])


def evaluate_one_sample(args):
    sample_name, experiment_name, sample_type, model_num = args

    experiment_dir = (
        Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/")
        / experiment_name
        / f"model_training_{model_num:03d}"
    )

    standardized_method_dict = auroc_utils.load_other_method_muon_grns([sample_name], sample_type)
    if not Path(experiment_dir / "inferred_grn.csv").exists():
        logging.warning(f"{experiment_name} does not have an inferred_grn.csv. Skipping Gradient Attribution for this sample.")
        return None
    
    grn_df = pd.read_csv(experiment_dir / "inferred_grn.csv", index_col=None)
    standardized_method_dict["Gradient Attribution"] = grn_df

    sample_results = []

    for gt_name, ground_truth in gt_by_dataset_dict[sample_type].items():
        for method_name, method_grn_df in standardized_method_dict.items():
            result = evaluate_single_method_setwise(
                method_name=method_name,
                score_df=method_grn_df,
                ground_truth=ground_truth,
                ground_truth_name=gt_name,
                max_edges=10000,
            )

            if result is not None:
                result["sample_name"] = sample_name
                result["sample_type"] = sample_type
                sample_results.append(result)

    if sample_results:
        return pd.concat(sample_results, ignore_index=True)

    return None

result_dfs = []

max_workers = 4

result_dfs = Parallel(n_jobs=4)(
    delayed(evaluate_one_sample)(exp)
    for exp in selected_preprocessing_experiments
)

result_dfs = [df for df in result_dfs if df is not None]
combined_df = pd.concat(result_dfs, ignore_index=True)
print(combined_df.head())

In [ ]:
from matplotlib.lines import Line2D

plot_df = combined_df.copy()

# Keep only methods that are actually present, in your desired order
methods_in_plot = plot_df["Method"].unique().tolist()
method_order = [m for m in exp.method_color_dict.keys() if m in methods_in_plot]

# Make sample order explicit and stable
sample_order = plot_df["sample_name"].drop_duplicates().tolist()

# Map sample names to base y positions
sample_to_y = {sample: i for i, sample in enumerate(sample_order)}
plot_df["y_base"] = plot_df["sample_name"].map(sample_to_y)

# Create fixed offsets for each method
offset_width = 0.15
n_methods = len(method_order)
offsets = np.linspace(
    -offset_width * (n_methods - 1) / 2,
    offset_width * (n_methods - 1) / 2,
    n_methods
)
method_to_offset = dict(zip(method_order, offsets))

plot_df["y_offset"] = plot_df["Method"].map(method_to_offset)
plot_df["y_position"] = plot_df["y_base"] + plot_df["y_offset"]

# Control draw order so earlier methods are plotted last / on top
plot_df["Method"] = pd.Categorical(plot_df["Method"], categories=method_order, ordered=True)
plot_df = plot_df.sort_values("Method", ascending=False)

metrics = ["Precision", "Recall", "F1_score"]

fig, axes = plt.subplots(
    ncols=len(metrics),
    nrows=1,
    figsize=(9, 6),
    sharey=True
)

# Clean y tick labels
y_ticks = list(sample_to_y.values())
y_tick_labels = [s.replace("_", " ") for s in sample_to_y.keys()]

for i, (ax, metric) in enumerate(zip(axes, metrics)):
    sns.scatterplot(
        data=plot_df,
        x=metric,
        y="y_position",
        hue="Method",
        hue_order=method_order,
        palette=method_color_dict,
        s=30,
        ax=ax,
        legend=False
    )

    # Horizontal separator lines between samples
    for y in range(len(sample_to_y) - 1):
        ax.axhline(y=y + 0.5, color="lightgray", linewidth=1, zorder=0)

    ax.set_title(metric.replace("_", " "))
    ax.grid(False)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    ax.invert_yaxis()

    # Set y-axis ticks/labels
    if i == 0:
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_tick_labels)
        ax.set_ylabel("")
    else:
        ax.set_yticks(y_ticks)   # keep alignment
        ax.set_yticklabels([])
        ax.set_ylabel("")

handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor=method_color_dict[m],
        markersize=6,
        label=m
    )
    for m in method_order
]

fig.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.00, 0.5),
    title="Methods",
    title_fontproperties={'weight': 'bold'}
)

axes[0].set_yticks(y_ticks)
axes[0].set_yticklabels(y_tick_labels)

fig.tight_layout()
plt.show()

In [ ]:

importlib.reload(data_formatter)
experiment_name = "Macrophage_buffer_2_auroc_by_kernel_size"
tdf = data_formatter.load_tdf(
    settings_path=f"/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA/HYPERPARAMETER_TUNING/{experiment_name}/settings.json",
)

In [ ]:
tdf.file_paths["processed"]

In [ ]:
importlib.reload(experiment_handler)
importlib.reload(data_formatter)

experiment_name = "Macrophage_buffer_2_auroc_by_kernel_size"
tdf = data_formatter.load_tdf(
    settings_path=f"/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA/HYPERPARAMETER_TUNING/{experiment_name}/settings.json",
)
exp = experiment_handler.load_experiment_handler(
    tdf_settings_path=tdf.settings_path,
    experiment_dir=Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/EXPERIMENTS/HYPERPARAMETER_TUNING"),
    model_num=1,
    )

In [ ]:
exp.batch_profile_log_df = pd.read_csv(exp.model_training_dir / "batch_profile_log.csv")
exp.batch_profile_log_df.head()

In [ ]:
exp.load_model()
exp.run_gradient_attribution(
    exp.test_loader,
)